## NEOs close-approaches clustering

Este proyecto tiene como objetivo aplicar técnicas modernas de machine learning, inteligencia artificial y algoritmos de clustering para analizar datos de Objetos Cercanos a la Tierra (NEOs). A través de estos enfoques, se busca identificar patrones comunes en sus características y realizar una segmentación basada en similitudes dentro del conjunto de datos.

# Data
Los datos fueron extraídos mediante el consumo de la API proporcionada por el sistema de monitoreo de aproximaciones cercanas a la Tierra (Close-Approach Data) del Center for Near Earth Object Studies, perteneciente a la NASA. La fuente oficial se encuentra disponible en: https://cneos.jpl.nasa.gov/ca/

## Librerias data

In [1]:
import pandas as pd
import os
import time
from datetime import datetime
import re
import requests
import math

## Carga y preparacion de datos

In [2]:
# Ruta colab
carpeta_destino = "."
os.makedirs(carpeta_destino, exist_ok=True)
ruta_csv = os.path.join(carpeta_destino, "close_approaches.csv")

In [3]:
# Funciones

def archivo_es_reciente(ruta, dias_maximos=30):
    """Devuelve True si el archivo fue modificado hace menos de dias_maximos días."""
    if not os.path.exists(ruta):
        return False
    edad_segundos = time.time() - os.path.getmtime(ruta)
    return edad_segundos < dias_maximos * 86400


def limpiar_fecha(fecha_str):
    """Elimina sufijos de incertidumbre como ±00:01 en las fechas de la API."""
    if isinstance(fecha_str, str):
        return fecha_str.split("±")[0].strip()
    return fecha_str

In [4]:
# Carga de datos
necesita_descarga = True
df = None

if os.path.exists(ruta_csv):
    print(f"✔ Archivo CSV encontrado en {ruta_csv}")
    if archivo_es_reciente(ruta_csv, dias_maximos=30):
        print("✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...")
        try:
            df = pd.read_csv(ruta_csv)
            print(f"✔ Datos cargados correctamente. {len(df):,} registros.")
            df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)
            necesita_descarga = False
        except Exception as e:
            print(f"✘ Error al cargar el archivo: {e}")
            print("  Se procederá a descargar datos frescos...")
            necesita_descarga = True
    else:
        print("✘ Archivo desactualizado (más de 30 días). Se descargará de nuevo.")
else:
    print(f"✘ Archivo CSV no encontrado en {ruta_csv}")

 # DESCARGA DESDE LA API DE JPL

if necesita_descarga:
    print("\n⬇ Descargando datos desde la API de JPL (puede tardar unos segundos)...")

    fecha_hoy = datetime.utcnow().strftime('%Y-%m-%d')

    url = (
        f"https://ssd-api.jpl.nasa.gov/cad.api"
        f"?date-min=1900-01-01"
        f"&date-max={fecha_hoy}"
        f"&diameter=true"
    )

    try:
        respuesta = requests.get(url, timeout=60)
        respuesta.raise_for_status()
        datos_json = respuesta.json()

        columnas = datos_json.get('fields', [])
        datos    = datos_json.get('data',   [])
        df = pd.DataFrame(data=datos, columns=columnas)
        print(f"✔ Descarga completada. {len(df):,} registros recibidos.")
        print(f"  Columnas disponibles: {list(df.columns)}")

        # Selección de columnas (solo las que existan en la respuesta)
        columnas_relevantes = ["des", "cd", "dist", "dist_min",
                               "v_rel", "v_inf", "h", "diameter", "diameter_sigma"]
        columnas_relevantes = [c for c in columnas_relevantes if c in df.columns]
        df = df[columnas_relevantes].copy()

        # Conversión numérica
        for col in ["dist", "dist_min", "v_rel", "v_inf", "h", "diameter", "diameter_sigma"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Diámetro estimado donde falta
        albedo = 1329 / math.sqrt(0.14)
        mascara_sin_diametro = df["diameter"].isna()
        df.loc[mascara_sin_diametro, "diameter"] = (
            albedo * (10 ** (-0.2 * df.loc[mascara_sin_diametro, "h"]))
        )
        df.loc[df["diameter_sigma"].isna(), "diameter_sigma"] = df["diameter"] * 0.35

        # Renombrar y guardar
        df = df.rename(columns={
            "des":            "Object",
            "cd":             "Close-Approach (CA) Date",
            "dist":           "CA DistanceNominal (au)",
            "dist_min":       "CA DistanceMinimum (au)",
            "v_rel":          "V relative(km/s)",
            "v_inf":          "V infinity(km/s)",
            "h":              "H(mag)",
            "diameter":       "Diameter(km)",
            "diameter_sigma": "Std Diameter(km)",
        })

        df.to_csv(ruta_csv, index=False)
        print(f"✔ CSV guardado en {ruta_csv}")

        df['Close-Approach (CA) Date'] = df['Close-Approach (CA) Date'].apply(limpiar_fecha)

    except requests.exceptions.RequestException as e:
        print(f"✘ Error de conexión: {e}")
        raise
    except Exception as e:
        print(f"✘ Error al procesar los datos: {e}")
        raise

if df is None:
    raise Exception("No se pudieron cargar los datos. Verifica la conexión o el archivo.")

print("\n           RESUMEN DE DATOS CARGADOS           ")
print(f"  Total de registros          : {len(df):,}")
print(f"  Nulos en H(mag)             : {df['H(mag)'].isna().sum():,}")
print(f"  Nulos en Diameter(km)       : {df['Diameter(km)'].isna().sum():,}")

print("\n            PRIMEROS 5 REGISTROS     ")
print(df[['Object', 'Diameter(km)', 'CA DistanceNominal (au)']].head().to_string(index=False))

if not necesita_descarga:
    print(f"\n Fuente: archivo local  →  {ruta_csv}")
    print(f"   Última modificación: {time.ctime(os.path.getmtime(ruta_csv))}")
else:
    print(f"\n Fuente: API de JPL (descarga fresca)")

✔ Archivo CSV encontrado en .\close_approaches.csv
✔ Archivo actualizado (menos de 30 días). Cargando desde archivo local...
✔ Datos cargados correctamente. 32,568 registros.

           RESUMEN DE DATOS CARGADOS           
  Total de registros          : 32,568
  Nulos en H(mag)             : 8
  Nulos en Diameter(km)       : 7

            PRIMEROS 5 REGISTROS     
    Object  Diameter(km)  CA DistanceNominal (au)
    509352      0.333013                 0.009632
2014 SC324      0.047039                 0.039964
2012 UK171      0.045547                 0.049706
  2024 BA5      0.022206                 0.026434
  2024 BW1      0.033609                 0.037979

 Fuente: archivo local  →  .\close_approaches.csv
   Última modificación: Thu Jun 18 19:27:57 2026


## Etiquetado de peligrosidad (PHA)

El objetivo del proyecto es **inferir el carácter potencialmente peligroso (PHA) de un NEO a partir únicamente de la cinemática de sus aproximaciones observadas**, sin usar los elementos orbitales que definen formalmente la etiqueta. Por eso construimos el *target* en dos versiones:

- **`PHA_official`** — el flag oficial `pha` de la [Small-Body Database (SBDB)](https://ssd-api.jpl.nasa.gov/doc/sbdb_query.html) de JPL. Es el *ground truth*. Un objeto es PHA si su **MOID ≤ 0.05 au** *y* su **magnitud absoluta H ≤ 22** (diámetro ≳ 140 m). Traemos además `MOID` y `H` oficiales **solo para validar y etiquetar**, nunca como variables predictoras.
- **`PHA_proxy`** — etiqueta derivada *exclusivamente de los datos observados*: el objeto se marca peligroso si su **H observado ≤ 22** y la **mínima distancia de aproximación observada (`dist_min`) ≤ 0.05 au**, usando la distancia observada como aproximación del MOID. La comparación `proxy` vs `official` mide cuán bien la distancia observada sustituye al MOID orbital (sub-resultado del paper).

> **Caveat documentado:** el diámetro fue imputado desde H con la relación de albedo (`albedo = 1329/√0.14`) en la celda anterior cuando faltaba; por eso `H(mag)` y `Diameter(km)` no son independientes para esos registros. Las etiquetas se calculan a nivel de **objeto** (no de evento) y se almacenan denormalizadas en el mismo CSV para no alterar la estructura de un solo archivo del pipeline.

> **Metadatos de selección (no son features):** también traemos de la SBDB la **fecha real de primera observación** (`first_obs`), el **arco orbital** (`data_arc`) y el número de observaciones (`n_obs_used`). Permiten un análisis de la función de selección observacional riguroso (por fecha de descubrimiento, no por año del primer evento de aproximación) y medir el confundidor de caracterización orbital.

In [5]:
# =====================================================================
#  ETIQUETADO DE PELIGROSIDAD (PHA)
#  Definicion oficial (IAU/CNEOS):  MOID <= 0.05 au  Y  H <= 22 mag
#  - PHA_official : flag oficial `pha` de la SBDB de JPL (ground truth).
#  - PHA_proxy    : derivado SOLO de lo observado (sin elementos orbitales),
#                   aproximando el MOID por la minima distancia observada.
#  Tambien traemos metadatos de OBSERVACION (no son features): la fecha real
#  de primera observacion y el arco orbital, para el analisis de seleccion.
# =====================================================================

ruta_sbdb = os.path.join(carpeta_destino, "sbdb_neo.csv")
sbdb_fields = ["pdes", "full_name", "pha", "neo", "moid", "H", "diameter",
               "first_obs", "last_obs", "data_arc", "n_obs_used"]

def _sbdb_cache_ok(ruta):
    """Cache valido solo si es reciente Y contiene todos los campos que pedimos."""
    if not archivo_es_reciente(ruta, dias_maximos=30):
        return False
    try:
        return set(sbdb_fields).issubset(pd.read_csv(ruta, nrows=1).columns)
    except Exception:
        return False

if _sbdb_cache_ok(ruta_sbdb):
    print(f"✔ Catalogo SBDB local reciente. Cargando {ruta_sbdb}...")
    sbdb = pd.read_csv(ruta_sbdb)
else:
    print("⬇ Descargando catalogo de NEOs desde la SBDB de JPL...")
    url_sbdb = "https://ssd-api.jpl.nasa.gov/sbdb_query.api"
    params_sbdb = {"fields": ",".join(sbdb_fields), "sb-group": "neo"}
    resp_sbdb = requests.get(url_sbdb, params=params_sbdb, timeout=120)
    resp_sbdb.raise_for_status()
    js = resp_sbdb.json()
    sbdb = pd.DataFrame(js["data"], columns=js["fields"])
    sbdb.to_csv(ruta_sbdb, index=False)
    print(f"✔ Catalogo SBDB guardado. {len(sbdb):,} NEOs.")

# --- Normalizacion de tipos y union por designacion (Object == pdes) ---
sbdb["pdes"] = sbdb["pdes"].astype(str)
for _c in ["moid", "H", "data_arc", "n_obs_used"]:
    sbdb[_c] = pd.to_numeric(sbdb[_c], errors="coerce")
sbdb["first_obs_year"] = pd.to_datetime(sbdb["first_obs"], errors="coerce").dt.year
df["Object"] = df["Object"].astype(str)
mapa = sbdb.drop_duplicates("pdes").set_index("pdes")

# Columnas oficiales: etiqueta + validacion + metadatos de SELECCION.
# NINGUNA se usa como feature predictora (eso seria circular).
df["MOID (au)"]      = df["Object"].map(mapa["moid"])
df["H_SBDB(mag)"]    = df["Object"].map(mapa["H"])
df["first_obs_year"] = df["Object"].map(mapa["first_obs_year"])  # fecha REAL de 1a observacion
df["data_arc(d)"]    = df["Object"].map(mapa["data_arc"])        # arco orbital (caracterizacion)
df["n_obs_used"]     = df["Object"].map(mapa["n_obs_used"])      # nro de observaciones astrometricas
df["PHA_official"]   = df["Object"].map(mapa["pha"]).map({"Y": 1, "N": 0}).astype("Int8")

# --- Etiqueta PROXY: propiedades por OBJETO desde lo observado ---
distmin_obj = df.groupby("Object")["CA DistanceMinimum (au)"].transform("min")
H_obj       = df.groupby("Object")["H(mag)"].transform("min")
df["PHA_proxy"] = ((H_obj <= 22) & (distmin_obj <= 0.05)).astype("Int8")

# --- Guardar CSV enriquecido (un unico archivo, estructura intacta) ---
df.to_csv(ruta_csv, index=False)
print(f"✔ CSV actualizado con etiquetas en {ruta_csv}")

# --- Snapshot CONGELADO y fechado para reproducibilidad del paper ---
import json as _json, sys as _sys
fecha_snap = datetime.utcnow().strftime("%Y%m%d")
ruta_snap = os.path.join(carpeta_destino, f"close_approaches_v{fecha_snap}.csv")
df.to_csv(ruta_snap, index=False)
_info = {"snapshot_date_utc": fecha_snap,
         "n_events": int(len(df)),
         "n_objects": int(df["Object"].nunique()),
         "n_pha_official": int((df.drop_duplicates("Object")["PHA_official"] == 1).sum()),
         "python": _sys.version.split()[0], "pandas": pd.__version__}
with open(os.path.join(carpeta_destino, "snapshot_info.json"), "w") as _fj:
    _json.dump(_info, _fj, indent=2)
print(f"✔ Snapshot congelado: {ruta_snap}  (ver snapshot_info.json)")

# --- Resumen / validacion fisica (a nivel OBJETO) ---
obj = df.drop_duplicates("Object")
n_match = int(obj["PHA_official"].notna().sum())
print("\n        RESUMEN DE ETIQUETADO (por objeto)        ")
print(f"  Objetos unicos              : {len(obj):,}")
print(f"  Emparejados con SBDB        : {n_match:,} ({100*n_match/len(obj):.1f}%)")
print(f"  PHA_official = 1            : {int((obj['PHA_official']==1).sum()):,}")
print(f"  PHA_proxy    = 1            : {int((obj['PHA_proxy']==1).sum()):,}")

val = obj.dropna(subset=["PHA_official"])
tp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==1)).sum())
fp = int(((val["PHA_proxy"]==1) & (val["PHA_official"]==0)).sum())
fn = int(((val["PHA_proxy"]==0) & (val["PHA_official"]==1)).sum())
prec = tp/(tp+fp) if tp+fp else float("nan")
rec  = tp/(tp+fn) if tp+fn else float("nan")
print(f"  Proxy vs oficial           : precision={prec:.3f}  recall={rec:.3f}")
vmoid = val.dropna(subset=["MOID (au)"])
corr = vmoid["CA DistanceMinimum (au)"].corr(vmoid["MOID (au)"])
print(f"  corr(dist_min observ., MOID): {corr:.3f}")

⬇ Descargando catalogo de NEOs desde la SBDB de JPL...


✔ Catalogo SBDB guardado. 42,074 NEOs.


✔ CSV actualizado con etiquetas en .\close_approaches.csv


✔ Snapshot congelado: .\close_approaches_v20260619.csv  (ver snapshot_info.json)

        RESUMEN DE ETIQUETADO (por objeto)        
  Objetos unicos              : 18,934
  Emparejados con SBDB        : 18,927 (100.0%)
  PHA_official = 1            : 1,379
  PHA_proxy    = 1            : 1,401
  Proxy vs oficial           : precision=0.968  recall=0.983
  corr(dist_min observ., MOID): 0.725


---

## Datos listos

El dataset procesado ha sido guardado como `close_approaches.csv` en esta misma carpeta (`data/`).

Para continuar con el analisis estadistico y machine learning, ejecutar el notebook `ProyectoNeoRework_ml.ipynb`.